In [ ]:
"""Python API analyzer."""

import ast
from pathlib import Path
from collections import defaultdict


def _annotate_parents(tree: ast.AST):
    """Add `.parent` links so we can walk up the tree."""
    for parent in ast.walk(tree):
        for child in ast.iter_child_nodes(parent):
            child.parent = parent


def _in_type_checking_block(node: ast.AST) -> bool:
    """Check if in an `if TYPE_CHECKING:` block."""
    while hasattr(node, "parent"):
        parent = node.parent
        if isinstance(parent, ast.If):
            if isinstance(parent.test, ast.Name) and parent.test.id == "TYPE_CHECKING":
                return True
        node = parent
    return False


class PmgAnalyzerPy(ast.NodeVisitor):
    """Python analyzer for pymatgen."""

    def __init__(self) -> None:
        # alias map: local name → full path
        self.aliases: dict[str, str] = {}
        # usage counts
        self.usage: dict[str, int] = defaultdict(int)

    def visit_Import(self, node):
        if _in_type_checking_block(node):
            return
        for alias in node.names:
            if alias.name.startswith("pymatgen"):
                asname = alias.asname or alias.name.split(".")[-1]
                self.aliases[asname] = alias.name

    def visit_ImportFrom(self, node):
        if _in_type_checking_block(node):
            return
        if node.module and node.module.startswith("pymatgen"):
            for alias in node.names:
                asname = alias.asname or alias.name
                self.aliases[asname] = f"{node.module}.{alias.name}"

    def visit_Call(self, node):
        """Track function/method calls on pymatgen objects."""
        if isinstance(node.func, ast.Attribute):
            base = node.func.value
            if isinstance(base, ast.Name) and base.id in self.aliases:
                full = f"{self.aliases[base.id]}.{node.func.attr}"
                self.usage[full] += 1
        self.generic_visit(node)


def analyze_py(path: str | Path) -> tuple[dict[str, str], dict[str, int]]:
    """
    Analyze a Python file for pymatgen API usage.

    Returns:
        aliases (dict): Mapping of local names → full pymatgen paths.
        usage (dict): Mapping of pymatgen API calls → count.
    """
    path = Path(path)

    if path.suffix != ".py":
        raise ValueError(f"cannot analyze non-py file: {path}")

    text = path.read_text(encoding="utf-8")

    tree = ast.parse(text)

    _annotate_parents(tree)

    analyzer = PmgAnalyzerPy()
    analyzer.visit(tree)
    return analyzer.aliases, dict(analyzer.usage)

In [ ]:
TEST_FILE = "./dependent-repos/pymatviz/pymatviz/chem_env.py"
aliases, usage = analyze_py(TEST_FILE)

print(aliases)
print(usage)

{'local_env': 'pymatgen.analysis.local_env'}
{'pymatgen.analysis.local_env.LocalStructOrderParams': 1, 'pymatgen.analysis.local_env.CrystalNN': 1}
